# Food Review Sentiment Classification & Public Health Surveillance
### B.Tech CSE Minor Project (ETMN100) — Amity University, Kolkata
**Author:** Tanim Naha  
**Objective:** Benchmark classical machine learning algorithms (Logistic Regression, Linear SVM, Multinomial Naive Bayes) for multi-class food review sentiment classification, and demonstrate an AI-driven early warning surveillance system for municipal food safety and hygiene violations.

## 1. Environment Setup and Library Imports

In [ ]:
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score

# Path to dataset
DATA_PATH = Path("../data/raw/yelp_sentiment_master_dataset.csv")
print("Libraries imported successfully.")

## 2. Dataset Loading and Exploratory Data Analysis (EDA)

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Total records loaded: {len(df):,}")
print("
Dataset Schema:")
df.info()
df.head(3)

In [ ]:
# Data Cleaning
df["text"] = df["text"].fillna("").astype(str).str.strip()
df["rating_review"] = df["rating_review"].fillna("").astype(str).str.strip().str.title()
LABELS = ["Negative", "Neutral", "Positive"]
df = df[df["text"].ne("") & df["rating_review"].isin(LABELS)].drop_duplicates(subset=["text"]).reset_index(drop=True)

print(f"Usable unique reviews: {len(df):,}")
print("
Class Distribution:")
print(df["rating_review"].value_counts())

## 3. Stratified Train-Test Split (80 / 20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["rating_review"],
    test_size=0.20,
    random_state=42,
    stratify=df["rating_review"]
)
print(f"Training set: {len(X_train):,} reviews")
print(f"Testing set:  {len(X_test):,} reviews")

## 4. Feature Extraction: TF-IDF Vectorizer with N-Grams (1, 2)

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000,
    sublinear_tf=True
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print(f"Vocabulary size: {len(vectorizer.get_feature_names_out()):,} features")

## 5. Model Training & Comparative Benchmarking

In [ ]:
models = {
    "MultinomialNB": MultinomialNB(),
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "LinearSVM": LinearSVC(class_weight="balanced")
}

results = []
for name, clf in models.items():
    clf.fit(X_train_vec, y_train)
    preds = clf.predict(X_test_vec)
    acc = accuracy_score(y_test, preds)
    macro_f1 = f1_score(y_test, preds, average="macro")
    weighted_f1 = f1_score(y_test, preds, average="weighted")
    results.append({"Model": name, "Accuracy": acc, "Macro F1": macro_f1, "Weighted F1": weighted_f1})
    print(f"{name:20s} | Accuracy: {acc:.4f} | Macro F1: {macro_f1:.4f}")

benchmark_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
benchmark_df

## 6. Public Health Surveillance: Food Safety & Hygiene Risk Flagging
Motivated by municipal food safety drives in Kolkata and West Bengal, this module classifies negative complaints into public health risk tiers.

In [ ]:
FOOD_SAFETY_TRIGGERS = {
    "Critical Biological / Illness Hazard": ["food poisoning", "vomiting", "stomach ache", "diarrhea", "nausea", "salmonella", "undercooked chicken"],
    "Contamination & Pest Infestation": ["cockroach", "rat", "mouse", "hair", "mold", "rancid", "stale oil", "spoiled"],
    "Kitchen Sanitation Violation": ["dirty table", "dirty utensils", "filthy", "no gloves", "bare hands", "dirty kitchen"]
}

def audit_food_safety(review):
    lowered = review.lower()
    findings = {}
    for cat, terms in FOOD_SAFETY_TRIGGERS.items():
        matches = [t for t in terms if re.search(r"" + re.escape(t) + r"", lowered)]
        if matches:
            findings[cat] = matches
    severity = "Critical" if "Critical Biological / Illness Hazard" in findings else ("High" if "Contamination & Pest Infestation" in findings else ("Moderate" if findings else "Safe"))
    return {"Severity": severity, "Flagged": len(findings) > 0, "Findings": findings}

sample = "The chicken was undercooked and I got violent food poisoning and vomiting!"
print("Audit Output:", audit_food_safety(sample))